# RiverWatch2 — CAMELS-531 no-q δHBV training (Kaggle GPU)

Trains **combined-loss δHBV** members (the decorrelation lever that was never actually trained into the shipped ckpts — `cfg.dhbv_loss=None`) at `nmul=16` + `--epochs 100`, the two recipe-fidelity fixes toward the Li/Shen 0.83 record.

**Session policy:** ~3 seeds/forcing per 12-hr session; each seed's `.pt` is saved on val-improve, then *Save Version* persists `/kaggle/working` to a Dataset (disk resets otherwise). Set `FORCING` + `SEEDS` below.


In [ ]:
# --- GPU init: verify CUDA is really there BEFORE spending a session ---
import subprocess
subprocess.run(['nvidia-smi'], check=False)
import torch
assert torch.cuda.is_available(), \
    'NO CUDA GPU — set Accelerator to GPU in the notebook settings!'
print('torch', torch.__version__, '| CUDA', torch.version.cuda,
      '| device', torch.cuda.get_device_name(0),
      '| capability', torch.cuda.get_device_capability(0))

# --- Setup: clone repo (SHA-logged). Kaggle has torch/pandas/numpy already ---
import os, sys, textwrap
os.chdir('/kaggle/working')
if not os.path.exists('riverwatch2'):
    subprocess.run(['git','clone','--depth','1','--branch','benchmark-competition-2026-07',
                    'https://github.com/andrewnakas/riverwatch2.git'], check=True)
os.chdir('/kaggle/working/riverwatch2')
sha = subprocess.run(['git','rev-parse','--short','HEAD'],
                     capture_output=True, text=True).stdout.strip()
print('repo SHA:', sha)
# Kaggle already has torch/pandas/numpy/scikit-learn; nothing else is needed.
os.environ['RW2_ENABLE_MBLSTM'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'


In [ ]:
# --- Wire Kaggle Dataset inputs to repo-relative paths ---
import glob, shutil, os
# Static attrs + gauge ids + station registry (load-bearing: without
# camels_attrs.json the static overlay is all-NaN and NSE craters to 0.40).
STATIC_DS = '/kaggle/input/rw2-camels-static'
for f in ['camels_attrs.json','camels_gauge_ids.json','stations_40_enriched.json']:
    src = os.path.join(STATIC_DS, f)
    if os.path.exists(src):
        shutil.copy(src, f'data/{f}')
        print('staged', f)
    else:
        print('WARNING missing static input:', src)
# Corpora: one Dataset per forcing. Resolve the dir that holds the 531 csv.gz.
def corpus_dir(forcing):
    cands = glob.glob(f'/kaggle/input/rw2-camels-corpus-{forcing}/**/camels_corpus_{forcing}_v2',
                      recursive=True) or \
            glob.glob(f'/kaggle/input/rw2-camels-corpus-{forcing}/**/*.csv.gz', recursive=True)
    if not cands: raise FileNotFoundError(f'no corpus for {forcing}')
    d = cands[0]
    return d if os.path.isdir(d) else os.path.dirname(d)
for F in ['daymet','maurer','nldas']:
    try: print(F, '->', corpus_dir(F), len(glob.glob(corpus_dir(F)+'/*.csv.gz')), 'basins')
    except Exception as e: print(F, 'NOT MOUNTED', e)


## Cell A — recipe-fidelity audit (NB0)
Diffs each shipped ckpt's cfg vs the paper recipe so we know exactly what each retrain must change. Cheap; no GPU.


In [ ]:
# Audit: what did the shipped members actually train with?
import torch, glob, json
rows = []
for f in sorted(glob.glob('/kaggle/input/**/camels531_*_dhbv_*.pt', recursive=True))[:6]:
    try:
        c = torch.load(f, map_location='cpu', weights_only=False)['cfg']
        rows.append({'file': f.split('/')[-1], 'dhbv_loss': c.get('dhbv_loss'),
                     'nmul': c.get('nmul'), 'q_transform': c.get('q_transform'),
                     'n_static': len(c.get('static_feats', [])),
                     'enc_vars': len(c.get('enc_vars', []))})
    except Exception as e: rows.append({'file': f, 'err': str(e)})
for r in rows: print(r)
print('\nTARGET: dhbv_loss=combined, nmul=16, epochs=100 (fidelity fixes)')


## Cell B — train (set FORCING + SEEDS)
Runs the recipe-v2 δHBV launch **plus** `--dhbv-loss combined --epochs 100`. One job per seed; `--init-ckpt` resumes a warm start if a session timed out.


In [ ]:
FORCING = 'daymet'          # daymet | maurer | nldas
SEEDS   = [971, 972, 973]   # 3 per 12-hr session is safe at 100 epochs

import subprocess, os
CORPUS = corpus_dir(FORCING)
os.makedirs('/kaggle/working/ckpts', exist_ok=True)
FLAGS = '--no-q-input --head dhbv --nmul 16 --dhbv-loss combined --forcing-correction --enc-vars camels1f --static-set camels --q-transform linear --hidden 256 --windows-per-station 1000 --batch 256 --val-stride 10 --train-start 1999-10-01 --train-end 2008-09-30 --val-start 1998-10-01 --val-end 1999-09-30 --epochs 100 --device cuda'
for s in SEEDS:
    out = f'/kaggle/working/ckpts/camels531_{FORCING}_dhbv_combined100_s{s}.pt'
    if os.path.exists(out):
        print('skip (exists)', out); continue
    cmd = (f'python scripts/train_mblstm.py --corpus-dir {CORPUS} '
           f'{FLAGS} --seed {s} --out {out}')
    print('>>', cmd, flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    print('seed', s, 'rc', rc, 'saved' if os.path.exists(out) else 'NO CKPT', flush=True)


## Cell C — persist weights
List the trained `.pt`; then **File → Save Version** (with *Save output*) so `/kaggle/working/ckpts` becomes the `rw2-noq-ckpts` Dataset for the dump notebook + local download.


In [ ]:
import glob, os
for f in sorted(glob.glob('/kaggle/working/ckpts/*.pt')):
    print(f, round(os.path.getsize(f)/1e6, 2), 'MB')
